# Baseline Comparison: Contrastive vs Linear Probe vs Token Entropy (Natural Questions)

This notebook trains and evaluates three hallucination detection methods on Natural Questions data:

1. **Contrastive** (ProgressiveCompressor + SupConLoss) — our primary method, evaluated via OOD metrics
2. **Linear Probe** (per-layer linear classifier with BCE loss) — learned baseline
3. **Token Entropy** (logprob statistics) — non-learned baseline

All methods share the same `ActivationParser` and underlying dataset.

**Primary metric**: AUROC

In [ ]:
import os
import sys
import glob
import re
import json
from pathlib import Path

import torch
import numpy as np
import pandas as pd
from torch.utils.data import DataLoader
from loguru import logger
from tqdm.auto import tqdm

from activation_logging.activation_parser import ActivationParser
from activation_research.model import ProgressiveCompressor, LinearProbe
from activation_research.trainer import (
    ContrastiveTrainer, ContrastiveTrainerConfig,
    LinearProbeTrainer, LinearProbeTrainerConfig,
)
from activation_research.token_entropy import TokenEntropyDetector
from activation_research.metric_evaluator import MultiMetricHallucinationEvaluator

logger.remove()
logger.add(sys.stdout, level="INFO")

In [ ]:
# ---- Paths (Natural Questions) ----
model_name = "Llama-3.1-8B-Instruct"
base_dir = Path("shared/natural_questions_logprob")
task_dir = base_dir / "natural_questions" / model_name

inference_json = str(task_dir / "generation.jsonl")
eval_json = str(task_dir / "eval_results.json")
raw_eval_jsonl = str(task_dir / "raw_eval_res.jsonl")
activations_path = str(base_dir / "activations.zarr")
eval_json_for_training = str(task_dir / "eval_results_for_training.json")

# ---- Dataset parameters ----
backend = "zarr"
relevant_layers = list(range(14, 30))
target_layers = [22, 26]
num_views = 4
outlier_class = 1
fixed_layer = None
response_logprobs_top_k = 20

# ---- Preload ----
preload = True
check_ram = True

# ---- Shared hyperparams ----
device = "auto"
input_dim = 4096
final_dim = 512

# Contrastive
contrastive_max_epochs = 50
contrastive_batch_size = 512
contrastive_lr = 1e-5
contrastive_temperature = 0.25
contrastive_steps_per_epoch = 200

# Linear probe
probe_max_epochs = 30
probe_batch_size = 512
probe_lr = 1e-3
probe_steps_per_epoch = 200

# Workers
num_workers = 30
persistent_workers = True

# Checkpoints
contrastive_checkpoint_dir = os.path.join("checkpoints", "baseline_cmp_contrastive_nq")
probe_checkpoint_dir = os.path.join("checkpoints", "baseline_cmp_linear_probe_nq")

print(f"Inference JSON     : {inference_json}")
print(f"Activations (zarr) : {activations_path}")

In [ ]:
# ---- Build ActivationParser-compatible eval JSON ----
def _load_jsonl(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return rows


def build_eval_for_activation_parser(
    inference_json_path, eval_json_path, raw_eval_jsonl_path, output_path,
):
    inference_path = Path(inference_json_path)
    if not inference_path.exists():
        raise FileNotFoundError(f"Missing inference file: {inference_path}")

    gendf = pd.read_json(inference_path, lines=True)
    n = len(gendf)
    if n == 0:
        raise RuntimeError("generation.jsonl is empty")

    halu, abstain, source_used = None, None, None

    eval_path = Path(eval_json_path)
    if eval_path.exists():
        payload = json.loads(eval_path.read_text(encoding="utf-8"))
        if (isinstance(payload, dict)
            and "halu_test_res" in payload and "abstantion" in payload
            and len(payload["halu_test_res"]) == n and len(payload["abstantion"]) == n):
            halu = [bool(x) for x in payload["halu_test_res"]]
            abstain = [bool(x) for x in payload["abstantion"]]
            source_used = "eval_results.json"

    if halu is None:
        raw_path = Path(raw_eval_jsonl_path)
        if raw_path.exists():
            raw_rows = _load_jsonl(raw_path)
            if len(raw_rows) == n:
                halu = [bool(r.get("is_hallucination", not bool(r.get("is_correct", False)))) for r in raw_rows]
                abstain = [False] * n
                source_used = "raw_eval_res.jsonl"

    if halu is None:
        halu = []
        for _, row in gendf.iterrows():
            answer = str(row.get("answer", "")).strip().lower()
            generation = str(row.get("generation", "")).strip().lower()
            halu.append(not (bool(answer) and answer in generation))
        abstain = [False] * n
        source_used = "substring fallback"

    hallu_count = int(sum(halu))
    compat = {
        "evaluator_abstantion": "natural_questions",
        "evaluator_hallucination": "natural_questions",
        "abstantion": abstain,
        "halu_test_res": halu,
        "total_count": n,
        "accurate_count": n - hallu_count,
        "hallu_count": hallu_count,
        "refusal_count": int(sum(abstain)),
        "correct_rate": float((n - hallu_count) / max(1, n)),
        "halu_rate_not_abstain": float(hallu_count / max(1, n - sum(abstain))),
        "refusal_rate": float(sum(abstain) / max(1, n)),
    }
    out = Path(output_path)
    out.parent.mkdir(parents=True, exist_ok=True)
    out.write_text(json.dumps(compat, indent=2), encoding="utf-8")
    print(f"Eval JSON written: {out}  (source: {source_used}, n={n}, halu={hallu_count})")
    return str(out)


if not Path(activations_path).exists():
    raise FileNotFoundError(f"Missing activations: {activations_path}")

eval_json_for_training = build_eval_for_activation_parser(
    inference_json, eval_json, raw_eval_jsonl, eval_json_for_training,
)

In [ ]:
# ---- Load datasets ----
ap = ActivationParser(
    inference_json=inference_json,
    eval_json=eval_json_for_training,
    activations_path=activations_path,
    logger_type="zarr",
    verbose=False,
)

# Contrastive needs multi-view; linear probe needs single-view; token entropy needs logprobs.
# We load the multi-view dataset for contrastive, then derive others.

train_dataset = ap.get_dataset(
    "train",
    relevant_layers=relevant_layers,
    fixed_layer=fixed_layer,
    num_views=num_views,
    backend=backend,
    include_response_logprobs=True,
    response_logprobs_top_k=response_logprobs_top_k,
    preload=preload,
    check_ram=check_ram,
)
test_dataset = ap.get_dataset(
    "test",
    relevant_layers=relevant_layers,
    fixed_layer=fixed_layer,
    num_views=num_views,
    backend=backend,
    include_response_logprobs=True,
    response_logprobs_top_k=response_logprobs_top_k,
    preload=preload,
    check_ram=check_ram,
)

print(f"train: {len(train_dataset)}")
print(f"test : {len(test_dataset)}")
print(ap.df["halu"].value_counts(dropna=False).rename(index={0: "non-halu", 1: "halu"}))

## 1. Contrastive Method (ProgressiveCompressor + SupConLoss)

In [ ]:
contrastive_model = ProgressiveCompressor(
    input_dim=input_dim,
    final_dim=final_dim,
    input_dropout=0.3,
)

contrastive_config = ContrastiveTrainerConfig(
    max_epochs=contrastive_max_epochs,
    batch_size=contrastive_batch_size,
    lr=contrastive_lr,
    temperature=contrastive_temperature,
    num_views=num_views,
    steps_per_epoch_override=contrastive_steps_per_epoch,
    device=device,
    num_workers=num_workers,
    persistent_workers=persistent_workers,
    checkpoint_dir=contrastive_checkpoint_dir,
    save_every=1,
    snapshot_every=10,
    snapshot_keep_last=5,
    use_labels=True,
    ignore_label=outlier_class,
    use_infinite_index_stream=True,
    use_infinite_index_stream_eval=True,
)

contrastive_trainer = ContrastiveTrainer(contrastive_model, config=contrastive_config)
contrastive_trainer.fit(train_dataset=train_dataset, val_dataset=test_dataset)

In [ ]:
# ---- Contrastive OOD evaluation ----
eval_device = device if device != "auto" else ("cuda" if torch.cuda.is_available() else "cpu")

train_dataset_target = train_dataset.slice_layers(target_layers)
test_dataset_target = test_dataset.slice_layers(target_layers)

train_loader_baseline = DataLoader(train_dataset_target, batch_size=64, shuffle=False)
eval_loader = DataLoader(test_dataset_target, batch_size=64, shuffle=False)

contrastive_model_eval = contrastive_trainer.model
contrastive_model_eval.eval()

ood_eval = MultiMetricHallucinationEvaluator(
    activation_parser_df=ap.df,
    train_data_loader=train_loader_baseline,
    layers=None,
    batch_size=256,
    sub_batch_size=64,
    device=str(contrastive_trainer.device),
    num_workers=num_workers,
    persistent_workers=False,
    outlier_class=outlier_class,
    metrics=[
        "cosine",
        "mds",
        {
            "metric": "knn",
            "kwargs": {
                "k": 50,
                "metric": "euclidean",
                "calibrate_k": True,
                "k_candidates": [50, 100, 200, 500, 1000],
                "max_train_size": 200000,
                "sample_seed": 0,
            },
            "train_selection": "all",
        },
    ],
)
contrastive_ood_stats = ood_eval.compute(eval_loader, contrastive_model_eval)
print("Contrastive OOD metrics:", contrastive_ood_stats)

## 2. Linear Probe (per-layer linear classifier)

Trains a single `LinearProbe` model (one linear layer + sigmoid) using BCE loss.
Uses `num_views=1` so each sample provides a single-layer activation slice.
We train on the same `relevant_layers` range and evaluate on `target_layers`.

In [ ]:
# ---- Linear probe datasets (num_views=1) ----
probe_train_dataset = ap.get_dataset(
    "train",
    relevant_layers=relevant_layers,
    fixed_layer=fixed_layer,
    num_views=1,
    backend=backend,
    preload=preload,
    check_ram=check_ram,
)
probe_test_dataset = ap.get_dataset(
    "test",
    relevant_layers=relevant_layers,
    fixed_layer=fixed_layer,
    num_views=1,
    backend=backend,
    preload=preload,
    check_ram=check_ram,
)

print(f"Probe train: {len(probe_train_dataset)}")
print(f"Probe test : {len(probe_test_dataset)}")

In [ ]:
# ---- Train linear probe ----
probe_model = LinearProbe(input_dim=input_dim, pooling="mean")

probe_config = LinearProbeTrainerConfig(
    max_epochs=probe_max_epochs,
    batch_size=probe_batch_size,
    lr=probe_lr,
    steps_per_epoch_override=probe_steps_per_epoch,
    device=device,
    num_workers=num_workers,
    persistent_workers=persistent_workers,
    checkpoint_dir=probe_checkpoint_dir,
    save_every=1,
    snapshot_every=10,
    snapshot_keep_last=5,
    pooling="mean",
    balanced_sampling=True,
)

probe_trainer = LinearProbeTrainer(probe_model, config=probe_config)
probe_trainer.fit(train_dataset=probe_train_dataset, val_dataset=probe_test_dataset)

In [ ]:
# ---- Linear probe final AUROC ----
# Run validation one more time to get the final AUROC
probe_val_stats = probe_trainer.validate(epoch=probe_max_epochs - 1, val_dataset=probe_test_dataset)
probe_auroc = probe_val_stats["val_auroc"]
probe_best_auroc = probe_val_stats["best_auroc"]
print(f"Linear Probe — final val AUROC: {probe_auroc:.4f}, best: {probe_best_auroc:.4f}")

## 3. Token Entropy (non-learned baseline)

Scores each sample using token-level logprob statistics: mean logprob, min logprob,
and approximate Shannon entropy from top-K logprobs. No training required.

In [ ]:
# ---- Token entropy evaluation ----
# Uses test_dataset which already has include_response_logprobs=True
entropy_detector = TokenEntropyDetector(outlier_class=outlier_class)
entropy_stats = entropy_detector.score(test_dataset, batch_size=256, num_workers=num_workers)

print("Token Entropy results:")
for k, v in entropy_stats.items():
    print(f"  {k}: {v:.4f}" if isinstance(v, float) else f"  {k}: {v}")

## Final Comparison

Side-by-side AUROC comparison across all methods.

In [ ]:
# ---- Build comparison table ----
rows = []

# Contrastive OOD metrics
rows.append({"Method": "Contrastive (cosine)", "AUROC": contrastive_ood_stats.get("cosine_auroc", float("nan"))})
rows.append({"Method": "Contrastive (Mahalanobis)", "AUROC": contrastive_ood_stats.get("mahalanobis_auroc", float("nan"))})
rows.append({"Method": "Contrastive (KNN)", "AUROC": contrastive_ood_stats.get("knn_auroc", float("nan"))})

# Linear probe
rows.append({"Method": "Linear Probe (BCE)", "AUROC": probe_best_auroc})

# Token entropy
rows.append({"Method": "Token Entropy (mean logprob)", "AUROC": entropy_stats.get("mean_logprob_auroc", float("nan"))})
rows.append({"Method": "Token Entropy (min logprob)", "AUROC": entropy_stats.get("min_logprob_auroc", float("nan"))})
if "mean_entropy_auroc" in entropy_stats:
    rows.append({"Method": "Token Entropy (Shannon)", "AUROC": entropy_stats["mean_entropy_auroc"]})

comparison_df = pd.DataFrame(rows).sort_values("AUROC", ascending=False).reset_index(drop=True)
print("=== AUROC Comparison ===")
comparison_df

In [ ]:
# ---- Bar chart ----
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 5))

colors = []
for method in comparison_df["Method"]:
    if "Contrastive" in method:
        colors.append("#2196F3")
    elif "Linear Probe" in method:
        colors.append("#FF9800")
    else:
        colors.append("#4CAF50")

bars = ax.barh(comparison_df["Method"], comparison_df["AUROC"], color=colors, edgecolor="white")
ax.set_xlabel("AUROC")
ax.set_title("Hallucination Detection — Method Comparison (Natural Questions)")
ax.set_xlim(0.4, 1.0)
ax.axvline(x=0.5, color="gray", linestyle="--", alpha=0.5, label="Random baseline")

for bar, val in zip(bars, comparison_df["AUROC"]):
    ax.text(bar.get_width() + 0.005, bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}", va="center", fontsize=10)

ax.legend()
plt.tight_layout()
plt.show()